In [0]:
import mlflow
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient

# 1. Target Unity Catalog
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

# 2. Use Unity Catalog's 3-level namespace format: catalog.schema.model_name
model_name = "health.default.healthcare_fraud_detector"
experiment_name = "/Users/sanyjain@asu.edu/healthcare_fraud_detection"
model_tmp_dir = "/Volumes/health/default/dataset/mlflow_tmp"
feature_cols = [
    "total_claims",
    "unique_patients",
    "avg_claim_amount",
    "claims_per_patient",
    "inpatient_ratio",
    "reimbursement_zscore",
    "cpp_zscore",
    "diagnosis_diversity_ratio"
]

# 3. Find the most recent logged fraud model from the training experiment
experiment = mlflow.get_experiment_by_name(experiment_name)
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["attributes.start_time DESC"],
    max_results=50,
    output_format="list"
)

source_model_uri = None
for run in runs:
    artifacts = client.list_artifacts(run.info.run_id)
    if any(artifact.path == "fraud_model" and artifact.is_dir for artifact in artifacts):
        source_model_uri = f"runs:/{run.info.run_id}/fraud_model"
        break

if source_model_uri is None:
    raise ValueError("No finished MLflow run with a fraud_model artifact was found in the training experiment.")

source_model = mlflow.spark.load_model(source_model_uri, dfs_tmpdir=model_tmp_dir)
sample_input_spark = spark.table("health.default.gold_provider_fraud_risk").select(*feature_cols).limit(5)
input_example = sample_input_spark.toPandas()
output_example = source_model.transform(sample_input_spark).select("prediction").toPandas()
signature = infer_signature(input_example, output_example)

with mlflow.start_run() as run:
    mlflow.spark.log_model(
        source_model,
        "fraud_model",
        signature=signature,
        input_example=input_example,
        dfs_tmpdir=model_tmp_dir
    )
    model_uri = f"runs:/{run.info.run_id}/fraud_model"
    model_details = mlflow.register_model(model_uri, model_name)

# 4. Promote to Production using an ALIAS instead of a STAGE
client.set_registered_model_alias(
    name=model_name,
    alias="production",
    version=model_details.version
)
print(f"Model version {model_details.version} successfully assigned 'production' alias.")

In [0]:
# Score new providers
import mlflow
from pyspark.sql import functions as F

mlflow.set_registry_uri("databricks-uc")
model_tmp_dir = "/Volumes/health/default/dataset/mlflow_tmp"
feature_cols = [
    "total_claims",
    "unique_patients",
    "avg_claim_amount",
    "claims_per_patient",
    "inpatient_ratio",
    "reimbursement_zscore",
    "cpp_zscore",
    "diagnosis_diversity_ratio"
]

def score_providers_batch():
    # Load production model
    model = mlflow.spark.load_model(
        "models:/health.default.healthcare_fraud_detector@production",
        dfs_tmpdir=model_tmp_dir
    )
    
    # Get new provider data
    new_providers = spark.table("health.default.gold_provider_fraud_risk") \
        .filter(F.col("Provider").isNotNull()) \
        .fillna(0, subset=feature_cols)
    
    # Score
    predictions = model.transform(new_providers)
    
    # Save results
    predictions.select(
        "Provider",
        "prediction",
        "probability",
        "total_claims",
        "total_reimbursement"
    ).write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("health.default.gold_fraud_predictions")
    
    # Flag high-risk providers
    high_risk = predictions.filter(F.col("prediction") == 1)
    print(f"High-risk providers detected: {high_risk.count()}")
    
    return predictions

predictions = score_providers_batch()